# Capa Gold - Agregaciones para analítica

Tablas analíticas optimizadas para consumo de BI.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, round as _round, desc

spark = SparkSession.builder.appName("GoldAgg").getOrCreate()

spark.sql("CREATE SCHEMA IF NOT EXISTS dbx_diplo_ricardo.gold_clase10")

## Lectura de Silver

In [ ]:
ventas_silver = spark.read.table("dbx_diplo_ricardo.silver_clase10.silver_ventas")

print("Filas Silver:", ventas_silver.count())
ventas_silver.printSchema()

## Ventas por fecha y tienda

In [ ]:
gold_ventas_tienda = (ventas_silver
                      .groupBy("fecha", "nombre_tienda")
                      .agg(
                          _sum("cantidad").alias("total_unidades"),
                          _round(_sum(col("cantidad") * col("precio_unitario")), 2).alias("total_ventas")
                      )
                      .orderBy("fecha", "nombre_tienda")
                     )

gold_ventas_tienda.show(10, truncate=False)

(gold_ventas_tienda.write
                   .format("delta")
                   .mode("overwrite")
                   .option("overwriteSchema", "true")
                   .saveAsTable("dbx_diplo_ricardo.gold_clase10.gold_ventas_por_tienda"))

## Ventas por categoría

In [ ]:
gold_ventas_categoria = (ventas_silver
                         .groupBy("categoria")
                         .agg(
                             _sum("cantidad").alias("total_unidades"),
                             _round(_sum(col("cantidad") * col("precio_unitario")), 2).alias("total_ventas")
                         )
                         .orderBy(desc("total_ventas"))
                        )

gold_ventas_categoria.show(truncate=False)

(gold_ventas_categoria.write
                      .format("delta")
                      .mode("overwrite")
                      .option("overwriteSchema", "true")
                      .saveAsTable("dbx_diplo_ricardo.gold_clase10.gold_ventas_por_categoria"))

## Top productos

In [ ]:
gold_top_productos = (ventas_silver
                      .groupBy("producto_id", "nombre_producto", "categoria")
                      .agg(
                          _sum("cantidad").alias("total_unidades"),
                          _round(_sum(col("cantidad") * col("precio_unitario")), 2).alias("total_ventas")
                      )
                      .orderBy(desc("total_ventas"))
                     )

gold_top_productos.show(10, truncate=False)

(gold_top_productos.write
                   .format("delta")
                   .mode("overwrite")
                   .option("overwriteSchema", "true")
                   .saveAsTable("dbx_diplo_ricardo.gold_clase10.gold_top_productos"))

## Verificación

In [ ]:
spark.sql("SHOW TABLES IN dbx_diplo_ricardo.gold_clase10").show(truncate=False)

In [ ]:
%sql
SELECT fecha, nombre_tienda, total_unidades, total_ventas
FROM dbx_diplo_ricardo.gold_clase10.gold_ventas_por_tienda
ORDER BY total_ventas DESC
LIMIT 10;

In [ ]:
%sql
SELECT * FROM dbx_diplo_ricardo.gold_clase10.gold_ventas_por_categoria;

In [ ]:
%sql
SELECT * FROM dbx_diplo_ricardo.gold_clase10.gold_top_productos LIMIT 10;